In [47]:
#Load data set from the google drive
from google.colab import drive
import pathlib

drive.mount('/content/drive')
!ls '/content/drive/MyDrive/MSC/DataSet/phm/'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
phm_test.csv  phm_train.csv  phm_train.gsheet


In [49]:
# Imports
import pandas as pd
import nltk
from nltk.corpus import stopwords
import numpy as np
import tensorflow as tf
import tensorflow as tf
from transformers import BertTokenizer, TFBertModel
from tensorflow.keras.layers import Input, Dropout, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model   # load saved model
from transformers import TFBertForSequenceClassification
from tensorflow.keras.layers import Layer

import re

In [50]:
# Introduced a BeartLayer here, this was because I was not able to use the classcle BERT implenetation using the keras. Seems better to use pytourch
class BertLayer(Layer):
    def __init__(self, model_name, **kwargs):
        super().__init__(**kwargs)
        self.bert = TFBertModel.from_pretrained(model_name)

    def call(self, inputs):
        input_ids, attention_mask = inputs
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token representation
        return cls_output

In [51]:
# Function load data from drive as csv
def load_from_csv():
    train_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_train.csv')
    test_data = pd.read_csv('/content/drive/MyDrive/MSC/DataSet/phm/phm_test.csv')

    print('\nData loaded..')

    return train_data, test_data


# Function for pre process data
def preprocess_dataset(tweet_data):
    x_data = tweet_data['tweet']       # Reviews/Input  --> Object
    y_data = tweet_data['label']    # Sentiment/Output  --> Int

    # PRE-PROCESS REVIEW
    nltk.download('stopwords')
    english_stops = set(stopwords.words('english'))

    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda tweet: [w for w in tweet.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda tweet: [w.lower() for w in tweet])   # lower case

    # y_data already encoded as 0 and 1 ( int values)
    print('\nData pre-process completed..')

    return x_data, y_data

# Function for get taining and testing data
def get_dataset(train_data, test_data):
    # remove tweet_id from df
    train_data = train_data.drop('tweet_id', axis=1)
    test_data = test_data.drop('tweet_id', axis=1)

    train_data = train_data[train_data['label'].isin([0, 1])]
    test_data = test_data[test_data['label'].isin([0, 1])]

    x_train, y_train = preprocess_dataset(train_data)
    x_test, y_test = preprocess_dataset(test_data)

    return x_train, y_train, x_test, y_test

# Function for getting the maximum tweet length
def get_max_length(x_train):
    tweet_length = []
    for tweet in x_train:
        length = len(tweet)
        tweet_length.append(length)

    return int(np.ceil(np.mean(tweet_length)))

# Function for BERT tokenization
def bert_tokenize(data, max_length, tokenizer):
    # Check if data is already tokenized (list of lists)
    if isinstance(data[0], list):
        print("Detected pre-tokenized input - joining tokens into text")
        data = [" ".join(tokens) for tokens in data]

    return tokenizer(
        data,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='tf'
    )

def tokenize_data(x_train, x_test, model):
    tokenizer = BertTokenizer.from_pretrained(model)

    max_length = min(get_max_length(x_train), 512)

    # Tokenize
    train_encodings = bert_tokenize(x_train, max_length, tokenizer)
    test_encodings = bert_tokenize(x_test, max_length, tokenizer)

    return train_encodings, test_encodings, max_length

# Build the BERT model
def build_bert_model(model_name, max_length, activation_fun, drop_out_rate, dense_activation_fun, learning_rate):
    # Input layers
    input_ids = Input(shape=(max_length,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(max_length,), dtype=tf.int32, name='attention_mask')

    cls_output = BertLayer(model_name)([input_ids, attention_mask])

    # Classification head
    x = Dropout(drop_out_rate)(cls_output)
    x = Dense(128, activation=activation_fun, kernel_regularizer=l2(L2_REG))(x)
    x = Dropout(drop_out_rate / 2)(x)
    output = Dense(1, activation=dense_activation_fun)(x)

    model = Model(inputs=[input_ids, attention_mask], outputs=output)

    # Compile
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model


# Evaluation function adapted for BERT
def evaluate_bert_repeatedly(model, train_dataset, test_dataset, n_runs=5, epochs=5, callbacks=None):
    all_accuracies = []
    for run in range(n_runs):
        print(f'\nRun {run+1}/{n_runs}')

        # Reset weights without rebuilding the whole model
        for layer in model.layers:
            if hasattr(layer, 'kernel_initializer'):
                layer.kernel.assign(layer.kernel_initializer(layer.kernel.shape))
            if hasattr(layer, 'bias_initializer'):
                layer.bias.assign(layer.bias_initializer(layer.bias.shape))

        history = model.fit(train_dataset,validation_data=test_dataset,epochs=epochs,callbacks=callbacks,verbose=1)

        test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
        all_accuracies.append(test_acc)
        print(f'Run {run+1} Test Accuracy: {test_acc:.4f}')

    return {
        "average_accuracy": sum(all_accuracies)/n_runs,
        "all_accuracies": all_accuracies
    }

In [53]:
# Load data
train_data, test_data = load_from_csv()
# Get preprocessed data
x_train, y_train, x_test, y_test = get_dataset(train_data, test_data)

# Hyperparameters (aligned with your LSTM setup where possible)
DROPOUT_RATE = 0.2
L2_REG = 0.001
LEARNING_RATE = 0.0005
BERT_MODEL_NAME = 'bert-base-uncased'
batch_size = 64


train_encodings, test_encodings, max_length = tokenize_data(x_train, x_test, BERT_MODEL_NAME)
MAX_LENGTH = max_length

#print(train_encodings)

# Convert to TensorFlow datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings),y_train)).batch(batch_size).prefetch(tf.data.AUTOTUNE)
test_dataset = tf.data.Dataset.from_tensor_slices((dict(test_encodings),y_test)).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Build the model with same parameters as before
model = build_bert_model(model_name=BERT_MODEL_NAME,max_length=MAX_LENGTH,activation_fun='relu',drop_out_rate=DROPOUT_RATE,dense_activation_fun='sigmoid',learning_rate=LEARNING_RATE)

# The rest of your training code remains the same
checkpoint = [
    ModelCheckpoint(
        'models/BERT_optimized.h5',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    )
]


print('\nModel Summary ---------------------------------------------------------')
print(model.summary())
print('\n')

n_runs = 1
epochs = 10

results = evaluate_bert_repeatedly(model, train_dataset, test_dataset, n_runs, epochs, checkpoint)

print('\n\n\nFinal Results -----------------------------------------------------')
average_accuracy_bert = results["average_accuracy"]
print('\nAverage Accuracy:', average_accuracy_bert)
print('All Accuracies:', results["all_accuracies"])


Data loaded..


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



Data pre-process completed..

Data pre-process completed..
Detected pre-tokenized input - joining tokens into text
Detected pre-tokenized input - joining tokens into text


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w


Model Summary ---------------------------------------------------------


...

157/157 ━━━━━━━━━━━━━━━━━━━━ 584s 4s/step - accuracy: 0.7304 - loss: 0.6613 - val_accuracy: 0.7649 - val_loss: 0.6289
Epoch 4/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7328 - loss: 0.6372
Epoch 4: val_accuracy improved from 0.76494 to 0.77154, saving model to models/BERT_optimized.h5


157/157 ━━━━━━━━━━━━━━━━━━━━ 675s 4s/step - accuracy: 0.7330 - loss: 0.6368 - val_accuracy: 0.7715 - val_loss: 0.6154
Epoch 5/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7365 - loss: 0.6240
Epoch 5: val_accuracy did not improve from 0.77154
157/157 ━━━━━━━━━━━━━━━━━━━━ 584s 4s/step - accuracy: 0.7367 - loss: 0.6237 - val_accuracy: 0.7682 - val_loss: 0.6112
Epoch 6/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7337 - loss: 0.6121
Epoch 6: val_accuracy improved from 0.77154 to 0.77214, saving model to models/BERT_optimized.h5


157/157 ━━━━━━━━━━━━━━━━━━━━ 585s 4s/step - accuracy: 0.7339 - loss: 0.6118 - val_accuracy: 0.7721 - val_loss: 0.6010
Epoch 7/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7352 - loss: 0.6046
Epoch 7: val_accuracy improved from 0.77214 to 0.77454, saving model to models/BERT_optimized.h5


157/157 ━━━━━━━━━━━━━━━━━━━━ 584s 4s/step - accuracy: 0.7354 - loss: 0.6042 - val_accuracy: 0.7745 - val_loss: 0.5850
Epoch 8/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7361 - loss: 0.5948
Epoch 8: val_accuracy improved from 0.77454 to 0.77604, saving model to models/BERT_optimized.h5


157/157 ━━━━━━━━━━━━━━━━━━━━ 640s 4s/step - accuracy: 0.7363 - loss: 0.5944 - val_accuracy: 0.7760 - val_loss: 0.5830
Epoch 9/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7377 - loss: 0.5910
Epoch 9: val_accuracy did not improve from 0.77604
157/157 ━━━━━━━━━━━━━━━━━━━━ 638s 4s/step - accuracy: 0.7379 - loss: 0.5907 - val_accuracy: 0.7760 - val_loss: 0.5848
Epoch 10/10
157/157 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7338 - loss: 0.5936
Epoch 10: val_accuracy did not improve from 0.77604
157/157 ━━━━━━━━━━━━━━━━━━━━ 585s 4s/step - accuracy: 0.7340 - loss: 0.5932 - val_accuracy: 0.7712 - val_loss: 0.5895
Run 1 Test Accuracy: 0.7760



Final Results -----------------------------------------------------

Average Accuracy: 0.7760432362556458
All Accuracies: [0.7760432362556458]
